In [2]:
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F

import logging

logger = logging.getLogger(__name__)

In [3]:
spark = SparkSession.builder.getOrCreate()

In [7]:
df = spark.read.csv('../data/raw/customers', header=True)

In [10]:
df.show()

+-----------+----------+---------+-------------+--------------------+------------+------------+------------+-----+--------+-------+--------------+------------+------------+
|customer_id|first_name|last_name|date_of_birth|               email|phone_number|     address|        city|state|zip_code|country|customer_since|credit_score|risk_segment|
+-----------+----------+---------+-------------+--------------------+------------+------------+------------+-----+--------+-------+--------------+------------+------------+
| CUST000000|    Mathew|   Nabers|   1973-09-29|mathew.nabers@exa...|555-782-6846|2759 Main St|    Columbus|   CA|   30196|    USA|    2015-09-19|         351|        High|
| CUST000001|      Mary|    Clark|   2004-09-21|mary.clark@exampl...|555-315-7610|5117 Main St|     Houston|   CA|   63282|    USA|    2024-09-16|         654|         Low|
| CUST000002|      Alex|   Carter|   1960-10-02|alex.carter@examp...|555-815-3744|9649 Main St|     Houston|   IL|   97436|    USA|    

In [9]:
df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- date_of_birth: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone_number: string (nullable = true)
 |-- address: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- zip_code: string (nullable = true)
 |-- country: string (nullable = true)
 |-- customer_since: string (nullable = true)
 |-- credit_score: string (nullable = true)
 |-- risk_segment: string (nullable = true)



In [16]:
def clean_customer_data(df: DataFrame) -> DataFrame:
    """ 
    Clean customers data by handling missing values and data types

    Args:
        df (DataFrame): Raw customers data
    
    Returns:
        DataFrame: Cleaned customers data
    
    """
    # Convert date strings to date format
    df = df.withColumn("date_of_birth", F.to_date("date_of_birth"))

    df = df.withColumn("customer_since", F.to_date("customer_since"))

    # Convert string to int
    df = df.withColumn("credit_score", F.col("credit_score").cast("int"))

    # Handle missing values 
    df = df.fillna("Unknown", ["city", "state", "country", "zip_code", "risk_segment"])

    return df


In [17]:
df1 = clean_customer_data(df)

In [20]:
df1.show()

+-----------+----------+---------+-------------+--------------------+------------+------------+------------+-----+--------+-------+--------------+------------+------------+
|customer_id|first_name|last_name|date_of_birth|               email|phone_number|     address|        city|state|zip_code|country|customer_since|credit_score|risk_segment|
+-----------+----------+---------+-------------+--------------------+------------+------------+------------+-----+--------+-------+--------------+------------+------------+
| CUST000000|    Mathew|   Nabers|   1973-09-29|mathew.nabers@exa...|555-782-6846|2759 Main St|    Columbus|   CA|   30196|    USA|    2015-09-19|         351|        High|
| CUST000001|      Mary|    Clark|   2004-09-21|mary.clark@exampl...|555-315-7610|5117 Main St|     Houston|   CA|   63282|    USA|    2024-09-16|         654|         Low|
| CUST000002|      Alex|   Carter|   1960-10-02|alex.carter@examp...|555-815-3744|9649 Main St|     Houston|   IL|   97436|    USA|    

In [39]:
def enrich_customer_data(df: DataFrame) -> DataFrame:
    """ 
    Enrich customer data

    Args:
        df (DataFrame): Raw customer data

    Returns:
        DataFrame: Enrcihed customer dataframe
    """
    logger.info("Enriching customer data")
    
    # Get full name
    df = df.withColumn("full_name", F.concat("first_name", F.lit(" "), "last_name"))
 
    # Combine address info to get full address
    df = df.withColumn("full_address", F.concat_ws(", ", "address", "city", "state", "zip_code", "country"))

    # Calculate the age
    df = df.withColumn("age", F.round(F.datediff(F.current_date(), "date_of_birth") / 365, 0).cast("int"))

    # Derive age group from age
    df = df.withColumn("age_group", F.when(F.col("age").between(18, 35), "18 to 35")
                                     .when(F.col("age").between(36, 50), "36 to 50")
                                     .when(F.col("age").between(51, 65), "51 to 65")
                                     .when(F.col("age") >= 66, "Over 66")
                                     .otherwise("underage") )

    return df


In [40]:
df2 = enrich_customer_data(df1)

In [41]:
df2.show(truncate=False)

+-----------+----------+---------+-------------+-------------------------+------------+------------+------------+-----+--------+-------+--------------+------------+------------+-------------+------------------------------------------+---+---------+
|customer_id|first_name|last_name|date_of_birth|email                    |phone_number|address     |city        |state|zip_code|country|customer_since|credit_score|risk_segment|full_name    |full_address                              |age|age_group|
+-----------+----------+---------+-------------+-------------------------+------------+------------+------------+-----+--------+-------+--------------+------------+------------+-------------+------------------------------------------+---+---------+
|CUST000000 |Mathew    |Nabers   |1973-09-29   |mathew.nabers@example.com|555-782-6846|2759 Main St|Columbus    |CA   |30196   |USA    |2015-09-19    |351         |High        |Mathew Nabers|2759 Main St, Columbus, CA, 30196, USA    |52 |51 to 65 |
|CUS

In [26]:
help(F.datediff)

Help on function datediff in module pyspark.sql.functions:

datediff(end: 'ColumnOrName', start: 'ColumnOrName') -> pyspark.sql.column.Column
    Returns the number of days from `start` to `end`.
    
    .. versionadded:: 1.5.0
    
    Examples
    --------
    >>> df = spark.createDataFrame([('2015-04-08','2015-05-10')], ['d1', 'd2'])
    >>> df.select(datediff(df.d2, df.d1).alias('diff')).collect()
    [Row(diff=32)]

